<a href="https://colab.research.google.com/github/MyosotisMatt/Scripts/blob/main/Sentinel1_Bicuar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This script accesses Earth Engine through the API and plots Sentinel 1 data

In [ ]:
import ee
import geemap
import json
from google.colab import files

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='ee-mathewrees54')


In [ ]:
Bicuar = ee.FeatureCollection('projects/ee-mathewrees54/assets/bicuar_limit_jose_luis_shp')

In [ ]:
imagecollection = (ee.ImageCollection('COPERNICUS/S1_GRD')
                 .filterBounds(Bicuar)
                 .filterDate('2020-01-01', '2026-03-01')
                 ##.filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                 .filter(ee.Filter.eq('instrumentMode', 'IW')) )

In [ ]:
count = imagecollection.select('VV').size().getInfo()
print(f'Number of images in collection: {count}')

if count > 0:
    # Get the date range of the available images
    dates = imagecollection.aggregate_array('system:time_start').getInfo()
    min_date = ee.Date(min(dates)).format('YYYY-MM-dd').getInfo()
    max_date = ee.Date(max(dates)).format('YYYY-MM-dd').getInfo()
    print(f'Date range: {min_date} to {max_date}')
else:
    print('The collection is empty. Try expanding your date range or removing filters.')

In [ ]:
print(imagecollection.mean().getInfo())

In [ ]:
m = geemap.Map()
m.centerObject(point, 12)

In [ ]:
vis_params = {
    'bands': ['VV', 'VH', 'VV'],
    'min': [-25, -30, -25],
    'max': [0, -10, 0]
}

m.addLayer(imagecollection.mean().clip(Bicuar), vis_params, 'Sentinel-1 False Color Composite')
m